<a href="https://colab.research.google.com/github/faithmutanu49/plp-da-a2-rainfall-advisory/blob/main/rainfall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Rainfall Analysis and Planting Advisory: Five Kenyan Counties

A farmer-support NGO sends SMS advice on planting windows. This notebook cleans a hand-typed
Machakos station record, combines it with four clean county records, analyses monthly rainfall
with NumPy, writes `rainfall_summary.csv` and ends with a short **Planting Advisory**.

Rainfall values are monthly totals in millimetres (mm), January to December.

In [4]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

months = np.array([
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
])

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

## 1. Clean the Machakos row

`clean_machakos` tries to convert each text reading to an integer. `int()` raises a `ValueError`
for text such as `"N/A"` or an empty string, so that is the only exception caught. Surrounding
spaces (`" 20 "`) are handled by `int()` itself, so that reading is valid. Invalid readings are
replaced with the rounded mean of the valid readings, and the imputed months are reported.

In [5]:
def clean_machakos(raw_values):
    """Convert raw text readings to ints and impute invalid ones.

    Invalid readings are replaced with the rounded mean of the valid readings.
    Returns (cleaned_values, imputed_indices, fill_value).
    """
    parsed = []
    imputed_idx = []
    for i, value in enumerate(raw_values):
        try:
            parsed.append(int(value))      # int(" 20 ") -> 20 (whitespace is stripped)
        except ValueError:
            parsed.append(None)            # "N/A" and "" cannot be converted
            imputed_idx.append(i)

    valid = [v for v in parsed if v is not None]
    if not valid:
        raise ValueError("No valid readings to compute an average from.")

    fill_value = round(sum(valid) / len(valid))
    cleaned = [fill_value if v is None else v for v in parsed]
    return cleaned, imputed_idx, fill_value


machakos_clean, imputed_idx, fill_value = clean_machakos(machakos_raw)

print("Cleaned Machakos row:", machakos_clean)
print(f"Imputed value: {fill_value} mm (rounded mean of the {12 - len(imputed_idx)} valid readings)")
print("Imputed months:", ", ".join(months[imputed_idx]))

Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Imputed value: 68 mm (rounded mean of the 10 valid readings)
Imputed months: April, September


Python's `round()` rounds halves to the nearest even number, so the valid mean of 67.5 mm becomes 68 mm,
which is also what conventional rounding gives here.

## 2. Combine into one 5 × 12 NumPy array

In [6]:
rainfall = np.array(clean_rows + [machakos_clean])

print("Shape:", rainfall.shape)
print("dtype:", rainfall.dtype)
assert rainfall.shape == (5, 12)
print()
print(rainfall)

Shape: (5, 12)
dtype: int64

[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]


## 3. Annual totals and monthly averages

`sum(axis=1)` collapses the months for each county (row totals). `mean(axis=0)` collapses the counties for
each month (column averages).

In [7]:
annual_totals = rainfall.sum(axis=1)
monthly_avg = rainfall.mean(axis=0)

print("Annual total rainfall (mm)")
for county, total in zip(counties, annual_totals):
    print(f"  {county:<9} {total:>5}")

print()
print("Average rainfall per month across all five counties (mm)")
for month, avg in zip(months, monthly_avg.round(1)):
    print(f"  {month:<9} {avg:>6.1f}")

Annual total rainfall (mm)
  Nairobi    1045
  Nakuru      916
  Kisumu     1371
  Mombasa    1075
  Machakos    811

Average rainfall per month across all five counties (mm)
  January     47.0
  February    47.6
  March       91.0
  April      164.4
  May        171.2
  June        67.6
  July        56.6
  August      58.2
  September   64.2
  October     67.8
  November   127.4
  December    80.6


## 4. Wettest month per county

`np.argmax(..., axis=1)` returns the column index of the maximum in each row. Indexing the `months`
array with those positions converts them to names.

In [8]:
wettest_idx = np.argmax(rainfall, axis=1)
wettest_month = months[wettest_idx]
wettest_mm = rainfall[np.arange(len(counties)), wettest_idx]

for county, month, mm in zip(counties, wettest_month, wettest_mm):
    print(f"{county:<9} wettest month: {month:<9} ({mm} mm)")

Nairobi   wettest month: April     (219 mm)
Nakuru    wettest month: April     (145 mm)
Kisumu    wettest month: April     (240 mm)
Mombasa   wettest month: May       (240 mm)
Machakos  wettest month: November  (182 mm)


## 5. Dry months (below 50 mm)

A boolean mask marks every reading under 50 mm. Summing the mask along `axis=1` counts the `True` values per county.

In [9]:
dry_mask = rainfall < 50
dry_counts = dry_mask.sum(axis=1)

for county, count, row_mask in zip(counties, dry_counts, dry_mask):
    names = ", ".join(months[row_mask]) if count else "none"
    print(f"{county:<9} {count} dry month(s): {names}")

Nairobi   4 dry month(s): June, July, August, September
Nakuru    3 dry month(s): January, February, December
Kisumu    0 dry month(s): none
Mombasa   2 dry month(s): January, February
Machakos  6 dry month(s): January, February, June, July, August, October


## 6. Planting window: the two consecutive months with the highest combined rainfall

Adding the array to a copy of itself shifted by one column (`R[:, :-1] + R[:, 1:]`) gives the combined rainfall
for every pair of neighbouring months (11 pairs per county). `argmax` over `axis=1` picks the best pair.

In [10]:
pair_totals = rainfall[:, :-1] + rainfall[:, 1:]      # shape (5, 11)
best_start = np.argmax(pair_totals, axis=1)
best_total = pair_totals[np.arange(len(counties)), best_start]

planting_window = np.char.add(np.char.add(months[best_start], "-"), months[best_start + 1])

print("pair_totals shape:", pair_totals.shape)
print()
for county, window, total in zip(counties, planting_window, best_total):
    print(f"{county:<9} planting window: {window:<14} ({total} mm combined)")

pair_totals shape: (5, 11)

Nairobi   planting window: April-May      (395 mm combined)
Nakuru    planting window: April-May      (265 mm combined)
Kisumu    planting window: April-May      (420 mm combined)
Mombasa   planting window: April-May      (390 mm combined)
Machakos  planting window: November-December (278 mm combined)


## 7. Write `rainfall_summary.csv`

The file is written with the `csv` module: one header row and five county rows. The NumPy results are converted
to plain Python values with `.tolist()` first.

In [11]:
header = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(zip(
        counties,
        annual_totals.tolist(),
        wettest_month.tolist(),
        dry_counts.tolist(),
        planting_window.tolist(),
    ))

# Read the file back to confirm it is readable and has exactly five data rows.
with open("rainfall_summary.csv", newline="") as f:
    rows = list(csv.reader(f))

assert rows[0] == header and len(rows) == 6
for row in rows:
    print(row)

['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']
['Nairobi', '1045', 'April', '4', 'April-May']
['Nakuru', '916', 'April', '3', 'April-May']
['Kisumu', '1371', 'April', '0', 'April-May']
['Mombasa', '1075', 'May', '2', 'April-May']
['Machakos', '811', 'November', '6', 'November-December']


## Checking how much the imputation matters

Two of the twelve Machakos readings were estimated. This cell checks how much of the Machakos total they make up
and whether a different true April value could change the recommended planting window.

In [12]:
m = counties.index("Machakos")

imputed_mm = fill_value * len(imputed_idx)
share = 100 * imputed_mm / annual_totals[m]
print(f"Imputed rainfall is {imputed_mm} mm of the {annual_totals[m]} mm Machakos total ({share:.1f}%).")

# April (index 3) was imputed. April-May would beat the chosen window only if April were large enough.
may_mm = rainfall[m, 4]
needed_april = best_total[m] - may_mm
print(f"Chosen window {planting_window[m]} totals {best_total[m]} mm.")
print(f"April-May would only win if the true April reading were above {needed_april} mm (imputed value: {rainfall[m, 3]} mm).")

Imputed rainfall is 136 mm of the 811 mm Machakos total (16.8%).
Chosen window November-December totals 278 mm.
April-May would only win if the true April reading were above 138 mm (imputed value: 68 mm).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Planting Advisory

Farmers in Nairobi, Nakuru, Kisumu and Mombasa should plan to plant in April–May, when the two wettest consecutive months bring between 265 mm (Nakuru) and 420 mm (Kisumu) of rain. Machakos farmers should wait for the short rains and plant in November–December (278 mm), which is that county's wettest two-month stretch. Kisumu has no month below 50 mm, while Nairobi has four dry months (June–September) and Machakos has six, so farmers in drier counties should expect to rely on one main planting season. The Machakos record is the least certain: April and September were unreadable and were filled with the 68 mm average, so its annual total (811 mm) and dry-month count are estimates, although its November–December window would only change if the true April rainfall were above 138 mm. These figures are monthly totals from one station per county and do not show how evenly rain falls within a month, so SMS messages should tell farmers to confirm the start of the rains with their local forecast before sowing.

## Data Quality Notes

- **Hand-typed Machakos data:** two of the 12 readings could not be converted (`"N/A"` for April and an empty entry for September). The reading `" 20 "` for June had stray spaces, which `int()` handled, so it was kept as 20 mm.
- **Imputation:** both gaps were filled with 68 mm, the rounded mean of the 10 valid readings (67.5 mm). This adds 136 mm, about 17% of the Machakos annual total, so that total is an estimate.
- **Dry-month count may be understated for Machakos:** both imputed values (68 mm) are above the 50 mm threshold. If the true April or September reading was below 50 mm, the Machakos count of six dry months would be higher.
- **Planting window sensitivity:** the Machakos November–December window changes only if the true April reading was above 138 mm.
- **Limited context:** the dataset does not say which year or years the readings cover, whether they are long-term averages or a single season, or how many stations feed each county figure. The four clean rows were used as given and were not independently checked.
- **Monthly totals only:** the data cannot show rain onset, dry spells within a month or rainfall intensity, so the planting windows are a guide rather than exact sowing dates.